In [ ]:
import sys
sys.path.append('./src')
import utils
import pandas as pd
import s3fs
import geopandas as gpd

arbres = pd.read_csv("https://www.data.gouv.fr/api/1/datasets/r/60433484-f30e-44ef-a362-e5553a9b7a42", sep = ";")

fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

MY_BUCKET = "raphcrre"
PATH_IRIS = f"{MY_BUCKET}/diffusion/projet_arbres/iris.gpkg"

with fs.open(PATH_IRIS, 'rb') as f:
    iris_france = gpd.read_file(f)

iris = iris_france[iris_france['code_insee'].astype(str).str.startswith('75')].copy()

df_arbres = utils.jointure_arbres_iris(arbres, iris)

df_arbres

Top 10 des essences (types) d'arbres à Paris 

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import utils3

# 1. Récupération des données via ton module
df_top = utils3.get_top_species(arbres, column_name = "LIBELLE FRANCAIS")

# 2. Création du graphique
plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

ax = sns.barplot(
    data=df_top, 
    x='Nombre', 
    y='Espece', 
    palette='viridis'
)

# 3. Cosmétique (pour que ce soit "qualitatif")
plt.title('Top 10 des essences d\'arbres à Paris', fontsize=15, pad=20)
plt.xlabel('Nombre d\'arbres')
plt.ylabel('Essence (nom commun)')

# Ajout des pourcentages au bout des barres
for i, p in enumerate(ax.patches):
    percentage = f"{df_top['Pourcentage'].iloc[i]}%"
    ax.annotate(percentage, (p.get_width(), p.get_y() + p.get_height()/2),
                ha='left', va='center', xytext=(5, 0), textcoords='offset points')

plt.show()

ModuleNotFoundError: No module named 'utils3'

Répartition du stade de développement des arbres 

In [ ]:
import matplotlib.pyplot as plt

# 1. Calcul
df_stade = utils3.get_development_stats(arbres)

# 2. Visualisation (Donut Chart)
plt.figure(figsize=(8, 8))
colors = ['#66b3ff', '#99ff99', '#ffcc99', '#ff9999'] # Couleurs douces

plt.pie(
    df_stade['Proportion'], 
    labels=df_stade['Stade'], 
    autopct='%1.1f%%', 
    startangle=140, 
    colors=colors,
    pctdistance=0.85
)

# Dessiner un cercle blanc au milieu pour faire un "Donut"
centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)

plt.title("Répartition des stades de développement des arbres à Paris", fontsize=14)
plt.tight_layout()
plt.show()

Carte des stades de développement des arbres par quartiers/iris 

In [ ]:
# Affiche la liste complète des colonnes et le type de données
print("--- COLONNES DISPONIBLES ---")
print(df_arbres.columns.tolist())

print("\n--- APERÇU DES ARRONDISSEMENTS (5 premières valeurs) ---")
print(df_arbres['ARRONDISSEMENT'].unique()[:5])

print("\n--- INFOS GÉNÉRALES ---")
df_arbres.info()

In [ ]:
%pip install folium

On calcule pour chaque IRIS le stade dominant (Jeune / Adulte / Vieux) ainsi que la proportion de chaque stade.
La colonne 'STADE DE DEVELOPPEMENT' contenait des valeurs hétérogènes (ex: 'Jeune (arbre)Adulte', 'Mature') qui ont été normalisées dans STADE_MAP de utils3.py. Sans ce mapping, ~40% des arbres tombaient en NaN et étaient silencieusement exclus des calculs.

In [ ]:
import sys
sys.path.append('./src')
import utils3
import geopandas as gpd

# ── 1. Préparer les géométries des IRIS (déjà chargées) ──────────────────────
# `iris` = GeoDataFrame des iris de Paris (code_insee commençant par '75')
# `df_arbres` = DataFrame issu de jointure_arbres_iris

# ── 2. Statistiques par IRIS ─────────────────────────────────────────────────
gdf_iris_stats = utils3.preparer_geodata_stade(iris, df_arbres, niveau='iris')
gdf_iris_stats[['code_iris', 'nom_iris', 'stade_dominant', 'pct_Jeune', 'pct_Adulte', 'pct_Vieux']].head()

Problème rencontré : les deux sources de données utilisent des conventions différentes pour nommer les arrondissements.
 - dataset arbres  → 'PARIS 13E ARRDT'  (libellé textuel majuscules)
 - fichier iris    → code_insee '75113' (code numérique IGN)
Un simple dissolve sur code_insee ou nom_commune ne suffisait pas car le merge ne trouvait aucune correspondance → toutes les stats à 0.
Solution : mapping explicite code_insee → libellé pour aligner les deux sources.

On reconstruit les polygones d'arrondissement en dissolvant les IRIS.
On filtre les IRIS sans correspondance dans le mapping pour exclure les zones hors arrondissements officiels (Bois de Vincennes, Bois de Boulogne) qui existent dans le fichier IGN mais pas comme arrondissements dans le dataset arbres.

Calcul des statistiques de stade par arrondissement.
NB : les arbres vieux (Mature dans les données source) ne représentent que ~4% du total et ne dominent aucun arrondissement — la couleur bleu foncé n'apparaît donc pas à ce niveau de zoom, ce qui est fidèle à la réalité.

In [ ]:
# Mapping code_insee → libellé ARRONDISSEMENT
mapping_arrdt = {
    '75101': 'PARIS 1ER ARRDT',
    '75102': 'PARIS 2E ARRDT',
    '75103': 'PARIS 3E ARRDT',
    '75104': 'PARIS 4E ARRDT',
    '75105': 'PARIS 5E ARRDT',
    '75106': 'PARIS 6E ARRDT',
    '75107': 'PARIS 7E ARRDT',
    '75108': 'PARIS 8E ARRDT',
    '75109': 'PARIS 9E ARRDT',
    '75110': 'PARIS 10E ARRDT',
    '75111': 'PARIS 11E ARRDT',
    '75112': 'PARIS 12E ARRDT',
    '75113': 'PARIS 13E ARRDT',
    '75114': 'PARIS 14E ARRDT',
    '75115': 'PARIS 15E ARRDT',
    '75116': 'PARIS 16E ARRDT',
    '75117': 'PARIS 17E ARRDT',
    '75118': 'PARIS 18E ARRDT',
    '75119': 'PARIS 19E ARRDT',
    '75120': 'PARIS 20E ARRDT',
}

iris_avec_arrdt = iris.copy()
iris_avec_arrdt['ARRONDISSEMENT'] = iris_avec_arrdt['code_insee'].map(mapping_arrdt)

# On ne garde que les vrais arrondissements (exclut Bois de Vincennes etc.)
iris_avec_arrdt = iris_avec_arrdt[iris_avec_arrdt['ARRONDISSEMENT'].notna()]

# Dissolve → un polygone par arrondissement
gdf_arrdt = iris_avec_arrdt.dissolve(by='ARRONDISSEMENT').reset_index()[['ARRONDISSEMENT', 'geometry']]

# Stats et carte
gdf_arrdt_stats = utils3.preparer_geodata_stade(gdf_arrdt, df_arbres, niveau='arrondissement')

# Vérification
print(gdf_arrdt_stats[['ARRONDISSEMENT', 'stade_dominant', 'total', 'pct_Jeune', 'pct_Adulte', 'pct_Vieux']])

 Génération de la carte interactive Folium avec deux niveaux de lecture :
  - Couche IRIS        : active par défaut, granularité fine
  - Couche Arrondissement : activable via le contrôle en haut à droite, vue d'ensemble
Survoler une zone affiche un tooltip rapide (stade dominant + proportions).
Cliquer sur une zone ouvre un popup détaillé avec les effectifs et pourcentages exacts par stade — utile pour compenser le fait que le bleu foncé (Vieux) est quasi absent visuellement vu la faible proportion de ces arbres.

In [ ]:
# Carte avec les deux niveaux
carte = utils3.carte_stade_paris(
    gdf_iris_stats=gdf_iris_stats,
    gdf_arrdt_stats=gdf_arrdt_stats,
    titre="Stade dominant des arbres à Paris"
)
carte

In [ ]:
# IRIS sans aucun arbre
iris_vides = gdf_iris_stats[gdf_iris_stats['total'] == 0]
print(f"{len(iris_vides)} IRIS sans arbres")
print(iris_vides[['nom_iris', 'code_iris', 'type_iris']].head(20))

Les 20 IRIS vides se répartissent en 3 catégories bien distinctes :

Type A (7 IRIS) → zones d'activités commerciales/tertiaires (Palais Royal, Madeleine, Chaussée d'Antin...). Pas d'arbres municipaux recensés, ce qui est logique.
Type D (1 IRIS) → Tuileries, c'est une zone de grande emprise (le jardin des Tuileries). Paradoxalement sans données, probablement parce que les arbres des jardins historiques gérés par l'État ne sont pas dans le dataset municipal de Paris.
Type H (12 IRIS) → zones à forte emprise d'équipements publics (hôpitaux, casernes, universités...). Même raison : les arbres de ces emprises ne sont pas gérés par la ville de Paris donc absents du dataset.

C'est une limite connue du dataset arbres.data.gouv.fr qui ne recense que les arbres gérés par la Direction des Espaces Verts de la Ville de Paris, et exclut donc les arbres appartenant à l'État, aux hôpitaux, aux universités, etc. 

Carte des densités et arbes remarquables

In [ ]:
importlib.reload(utils3)

# Densité par IRIS
gdf_iris_densite = utils3.calculer_densite_par_zone(df_arbres, iris, groupby_col='code_iris')

# Densité par arrondissement (réutilise gdf_arrdt construit précédemment)
gdf_arrdt_densite = utils3.calculer_densite_par_zone(df_arbres, gdf_arrdt, groupby_col='ARRONDISSEMENT')

# Arbres remarquables (top 50 hauteur + top 50 circonférence + colonne REMARQUABLE)
gdf_remarquables = utils3.extraire_arbres_remarquables(df_arbres, top_n_hauteur=50, top_n_circonf=50)
print(f"{len(gdf_remarquables)} arbres remarquables extraits")
print(gdf_remarquables[['LIBELLE FRANCAIS', 'HAUTEUR (m)', 'CIRCONFERENCE (cm)', 'motif_remarquable']].head())

In [ ]:
# Carte densité + arbres remarquables
carte_densite = utils3.carte_densite_paris(
    gdf_iris_densite=gdf_iris_densite,
    gdf_arrdt_densite=gdf_arrdt_densite,
    gdf_remarquables=gdf_remarquables,
    titre="Densité d'arbres à Paris"
)
carte_densite